# Trend following sobre series de demanda

Llevar el instrumental de momentum de trading a la demanda: ¿hay tendencias que
persisten y se pueden seguir, o los movimientos se dan vuelta?

## El orden importa

Es tentador armar la batería de señales — cruces de medias, breakouts, MACD — y
tirarlas al modelo a ver cuál pega. Ese es el camino corto al *overfitting*: con
suficientes señales, alguna va a correlacionar con el target por azar.

Así que primero se testea si **existe algo que seguir**, y recién después se
construyen las señales:

1. **¿Momentum o reversión?** Autocorrelación de los *cambios* y test de
   *variance ratio*. Si los cambios se autocorrelacionan positivo, hay tendencia
   que persiste. Si negativo, la estrategia correcta es la contraria.
2. **Batería de señales**, sólo si el paso 1 dio algo.
3. **Information Coefficient** de cada señal: correlación de rango entre la señal
   en `t` y el cambio efectivo en `t+2`. Es como se evalúa un factor en quant, y
   sirve de filtro barato **antes** de meter nada al modelo.
4. **Backtest**: ¿ajustar el naive con la señal baja el WAPE?

## Dos advertencias que condicionan todo

**La estacionalidad falsea el momentum.** Si todos los años sube en octubre, una
serie cruda va a mostrar "tendencia" en septiembre que no es más que el calendario.
Todas las señales se calculan sobre la serie **desestacionalizada**.

**Ya hay trend following en el pipe.** `02_FE` calcula `B1` (pendiente de la recta
ajustada a la ventana de lags) y `tn_racha` (meses consecutivos subiendo o bajando).
`B1` resultó ser la feature más importante del modelo — el 26,5% del gain. Así que
la pregunta acá no es "¿sirve la tendencia?" sino **"¿qué agrega una señal nueva
sobre `B1`, que ya está?"**. Por eso el IC se reporta también *residualizado*
contra `B1`.

## 0 — Ambiente

In [ ]:
import os, json, warnings
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env:
        return Path(env).expanduser().resolve()
    # ~/buckets/b1 primero: es donde lo monta la instalacion de la catedra,
    # y sirve para cualquier usuario (ds, natalialabo3, el que sea).
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


BUCKET  = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
DIR_OUT = BUCKET / "datasets_fe"
DIR_OUT.mkdir(parents=True, exist_ok=True)

SERIE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
         "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
TINTA, TINTA2, MUDO = "#0b0b0b", "#52514e", "#898781"
GRILLA, EJE_C, FONDO = "#e1e0d9", "#c3c2b7", "#fcfcfb"
POS, NEG = "#2a78d6", "#e34948"          # divergente: signo del IC

plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO,
    "axes.edgecolor": EJE_C, "axes.labelcolor": TINTA2,
    "text.color": TINTA, "xtick.color": MUDO, "ytick.color": MUDO,
    "grid.color": GRILLA, "grid.linewidth": .8,
    "axes.grid": True, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "figure.dpi": 110,
    "legend.frameon": False,
})

def limpiar(ax, titulo=None, y=None, x=None):
    if titulo: ax.set_title(titulo, color=TINTA, loc="left", pad=10)
    if y: ax.set_ylabel(y)
    if x: ax.set_xlabel(x)
    ax.grid(axis="x", visible=False)
    return ax

HORIZONTE = 2
MIN_MESES = 24
print(f"BUCKET: {BUCKET}")

## 1 — Panel y desestacionalización

In [ ]:
sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t").unique(subset=["product_id"])

def a_m(c): return (pl.col(c) // 100) * 12 + (pl.col(c) % 100)
def m_a_periodo(m): return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1

base = (sell.group_by(["product_id","periodo"]).agg(pl.col("tn").sum().alias("tn"))
            .with_columns(a_m("periodo").alias("m")))
vida = base.group_by("product_id").agg(pl.col("m").min().alias("m_nace"),
                                       pl.col("m").max().alias("m_muere"),
                                       pl.col("tn").sum().alias("tn_total"))
grilla = (vida.select("product_id","m_nace","m_muere")
              .with_columns(pl.int_ranges("m_nace", pl.col("m_muere")+1).alias("m"))
              .explode("m").drop("m_nace","m_muere"))
panel = (grilla.join(base, on=["product_id","m"], how="left")
               .with_columns(pl.col("tn").fill_null(0.0))
               .join(prod.select("product_id","cat1","cat2","cat3","brand"), on="product_id", how="left")
               .sort(["product_id","m"]))

# ── Indice estacional del mercado (media movil centrada de 12) ───────────
merc = panel.group_by("m").agg(pl.col("tn").sum().alias("tn")).sort("m")
merc = merc.with_columns(
    pl.col("tn").rolling_mean(12, center=True, min_periods=12).alias("tend"))
est = (merc.drop_nulls("tend")
           .with_columns((pl.col("tn")/pl.col("tend")).alias("ratio"),
                         ((pl.col("m")-1) % 12 + 1).alias("mes_cal"))
           .group_by("mes_cal").agg(pl.col("ratio").mean().alias("idx"))
           .sort("mes_cal"))
# normalizar para que el promedio del indice sea 1
est = est.with_columns((pl.col("idx")/est["idx"].mean()).alias("idx"))
IDX = dict(zip(est["mes_cal"].to_list(), est["idx"].to_list()))
print("indice estacional:", {k: round(v,3) for k,v in IDX.items()})

panel = (panel.with_columns(((pl.col("m")-1) % 12 + 1).alias("mes_cal"))
              .with_columns(pl.col("mes_cal").replace_strict(IDX, default=1.0).alias("idx_est"))
              .with_columns((pl.col("tn")/pl.col("idx_est")).alias("tn_sa")))
print(f"\npanel: {panel.height:,} filas · {panel['product_id'].n_unique()} productos")
print("tn_sa = tn desestacionalizada (dividida por el indice del mes)")

## 2 — ¿Momentum o reversión?

Dos tests clásicos, aplicados a los **cambios** de la serie desestacionalizada.

**Autocorrelación de los cambios.** Si `Δ(t)` correlaciona positivo con `Δ(t+k)`, lo
que subió tiende a seguir subiendo — hay tendencia que persiste. Negativo significa
reversión: lo que subió tiende a corregir.

**Variance ratio** (Lo–MacKinlay). Si los cambios fueran independientes, la varianza
del cambio a `q` meses sería `q` veces la del cambio mensual. El cociente
`VR(q) = Var(Δ_q) / (q · Var(Δ_1))` mide la desviación:

- `VR > 1` → los cambios se refuerzan: **tendencia**
- `VR = 1` → paseo aleatorio, no hay nada que seguir
- `VR < 1` → los cambios se compensan: **reversión**

Se calcula a tres niveles de agregación, porque puede haber tendencia en la
categoría y ruido en el producto suelto.

In [ ]:
elegibles = (vida.filter((pl.col("m_muere")-pl.col("m_nace")+1) >= MIN_MESES)
                 .sort("tn_total", descending=True)["product_id"].to_list())
print(f"{len(elegibles)} productos con >= {MIN_MESES} meses")

def series_por_nivel():
    """Devuelve {nivel: [array, ...]} de series desestacionalizadas."""
    out = {}
    g = panel.filter(pl.col("product_id").is_in(elegibles)).sort(["product_id","m"])
    out["producto"] = [x["tn_sa"].to_numpy() for _, x in g.group_by("product_id", maintain_order=True)]
    c = panel.group_by(["cat3","m"]).agg(pl.col("tn_sa").sum().alias("tn_sa")).sort(["cat3","m"])
    out["cat3"] = [x["tn_sa"].to_numpy() for _, x in c.group_by("cat3", maintain_order=True)
                   if x.height >= MIN_MESES]
    t = panel.group_by("m").agg(pl.col("tn_sa").sum().alias("tn_sa")).sort("m")
    out["mercado"] = [t["tn_sa"].to_numpy()]
    return out

NIV = series_por_nivel()
print({k: len(v) for k, v in NIV.items()})


def autocorr_cambios(series, lags=6):
    """Autocorrelacion media de las primeras diferencias, por lag."""
    acc = {L: [] for L in range(1, lags+1)}
    for s in series:
        d = np.diff(s)
        if len(d) < lags + 6 or np.std(d) == 0:
            continue
        for L in range(1, lags+1):
            a, b = d[:-L], d[L:]
            if np.std(a) == 0 or np.std(b) == 0:
                continue
            acc[L].append(float(np.corrcoef(a, b)[0, 1]))
    return {L: (np.mean(v) if v else np.nan) for L, v in acc.items()}, \
           {L: (np.std(v)/np.sqrt(len(v)) if len(v) > 1 else np.nan) for L, v in acc.items()}


def variance_ratio(series, qs=(2,3,4,6)):
    out = {}
    for q in qs:
        vals = []
        for s in series:
            if len(s) < q*4:
                continue
            d1 = np.diff(s)
            dq = s[q:] - s[:-q]
            v1, vq = np.var(d1, ddof=1), np.var(dq, ddof=1)
            if v1 > 0:
                vals.append(vq / (q*v1))
        out[q] = float(np.median(vals)) if vals else np.nan
    return out

print("\n=== AUTOCORRELACION DE LOS CAMBIOS (media entre series) ===")
AC = {}
for niv, ss in NIV.items():
    mu, se = autocorr_cambios(ss)
    AC[niv] = (mu, se)
    print(f"{niv:9s} " + "  ".join(f"L{L}:{mu[L]:+.3f}" for L in sorted(mu)))

print("\n=== VARIANCE RATIO (mediana; >1 tendencia, <1 reversion) ===")
VR = {}
for niv, ss in NIV.items():
    VR[niv] = variance_ratio(ss)
    print(f"{niv:9s} " + "  ".join(f"q{q}:{v:.2f}" for q, v in VR[niv].items()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.6))

ax = axes[0]
for j, niv in enumerate(["producto", "cat3", "mercado"]):
    mu, se = AC[niv]
    L = sorted(mu); v = [mu[k] for k in L]
    ax.plot(L, v, color=SERIE[j], linewidth=2, marker="o", markersize=5)
    ax.annotate(niv, (L[-1], v[-1]), xytext=(6,0), textcoords="offset points",
                color=SERIE[j], fontsize=9, va="center")
ax.axhline(0, color=EJE_C, linewidth=1.2)
ax.set_xlim(.7, 7.2)
limpiar(ax, "Autocorrelación de los cambios mensuales", y="correlación", x="lag (meses)")
ax.annotate("arriba de 0 = momentum\nabajo = reversión", (.03,.06), xycoords="axes fraction",
            color=TINTA2, fontsize=8)

ax = axes[1]
for j, niv in enumerate(["producto", "cat3", "mercado"]):
    q = sorted(VR[niv]); v = [VR[niv][k] for k in q]
    ax.plot(q, v, color=SERIE[j], linewidth=2, marker="o", markersize=5)
    ax.annotate(niv, (q[-1], v[-1]), xytext=(6,0), textcoords="offset points",
                color=SERIE[j], fontsize=9, va="center")
ax.axhline(1, color=EJE_C, linewidth=1.2)
ax.set_xlim(1.7, 7)
limpiar(ax, "Variance ratio", y="VR(q)", x="horizonte q (meses)")
plt.tight_layout(); plt.show()

vr_prod = VR["producto"][HORIZONTE] if HORIZONTE in VR["producto"] else np.nan
ac1 = AC["producto"][0][1]
print("VEREDICTO")
if ac1 > .05 and vr_prod > 1.05:
    print(f"  MOMENTUM a nivel producto (autocorr L1 {ac1:+.3f}, VR {vr_prod:.2f}).")
elif ac1 < -.05 and vr_prod < .95:
    print(f"  REVERSION a nivel producto (autocorr L1 {ac1:+.3f}, VR {vr_prod:.2f}).")
    print()
    print("  CONSECUENCIA PRACTICA: trend following puro va en la direccion equivocada.")
    print("  La senial explotable es la CONTRARIA -- lo que subio mucho tiende a corregir.")
    print("  Las mismas seniales sirven, con el signo dado vuelta: el IC de la seccion 4")
    print("  lo va a mostrar como IC negativo, y un IC negativo fuerte es tan util como")
    print("  uno positivo (el modelo aprende el signo solo).")
    print()
    print("  Cuidado con una explicacion alternativa antes de festejar: parte de esta")
    print("  autocorrelacion negativa puede ser ruido de medicion. Si un pedido se")
    print("  adelanta o se atrasa de mes, un mes sube y el siguiente baja sin que haya")
    print("  ninguna dinamica economica de reversion. Con datos de sell-in mensual eso")
    print("  es habitual, y no se puede distinguir con esta serie sola.")
else:
    print(f"  SIN SENIAL CLARA a nivel producto (autocorr L1 {ac1:+.3f}, VR {vr_prod:.2f}).")

## 3 — Batería de señales

Adaptaciones directas del instrumental de trading. Todas se calculan **sólo con
información hasta `t`** y sobre la serie desestacionalizada.

| señal | equivalente en trading | qué mide |
|---|---|---|
| `mom_k` | momentum a k meses | cambio relativo contra hace `k` meses |
| `ma_cross` | cruce de medias | `MA3 / MA12 − 1`: rápida sobre lenta = tendencia alcista |
| `macd` | MACD normalizado | `(MA3 − MA6)` escalado por el nivel |
| `donchian` | canal de Donchian | posición dentro del rango de los últimos 12 meses, 0–1 |
| `tstat_slope` | fuerza de tendencia | pendiente de la recta ÷ su error estándar |
| `racha` | — | meses consecutivos subiendo (+) o bajando (−) |
| `vol` | volatilidad realizada | desvío de los cambios relativos: filtro de calidad |
| `z_vs_ma12` | desvío contra la media | `(tn − MA12) / σ12`: alto = estirado, candidato a reversión |

`tstat_slope` es primo de `B1` del pipe, con una diferencia: `B1` es la pendiente
cruda y ésta la divide por su incertidumbre. Una pendiente grande sobre una serie
ruidosa vale menos que una chica sobre una serie limpia.

In [ ]:
W = 12   # ventana larga

def señales_de_serie(v):
    """v = tn_sa ordenada. Devuelve dict de arrays, todos causales (solo pasado)."""
    n = len(v)
    out = {}
    def ma(k):
        c = np.convolve(v, np.ones(k), "full")[:n]
        cnt = np.convolve(np.ones(n), np.ones(k), "full")[:n]
        return c / cnt
    ma3, ma6, ma12 = ma(3), ma(6), ma(12)
    eps = 1e-9
    nivel = np.maximum(ma12, eps)

    for k in (3, 6, 12):
        prev = np.concatenate([np.full(k, np.nan), v[:-k]]) if k < n else np.full(n, np.nan)
        out[f"mom_{k}"] = (v - prev) / np.maximum(np.abs(prev), eps)

    out["ma_cross"] = ma3 / np.maximum(ma12, eps) - 1.0
    out["macd"]     = (ma3 - ma6) / nivel

    don = np.full(n, np.nan)
    for i in range(n):
        lo_i = max(0, i - W + 1)
        w = v[lo_i:i+1]
        rng = w.max() - w.min()
        don[i] = (v[i] - w.min()) / rng if rng > 0 else .5
    out["donchian"] = don

    ts = np.full(n, np.nan)
    for i in range(n):
        lo_i = max(0, i - W + 1)
        w = v[lo_i:i+1]
        if len(w) >= 6 and np.std(w) > 0:
            x = np.arange(len(w), dtype=float)
            b, a = np.polyfit(x, w, 1)
            resid = w - (a + b*x)
            se = np.sqrt((resid**2).sum() / max(len(w)-2, 1) / max(((x-x.mean())**2).sum(), eps))
            ts[i] = b / se if se > 0 else 0.0
    out["tstat_slope"] = ts

    d = np.diff(v, prepend=v[0])
    racha = np.zeros(n)
    for i in range(1, n):
        s = np.sign(d[i])
        racha[i] = (racha[i-1] + s) if (s != 0 and np.sign(racha[i-1]) == s) else s
    out["racha"] = racha

    ret = d / np.maximum(np.abs(np.concatenate([[v[0]], v[:-1]])), eps)
    vol = np.full(n, np.nan)
    z   = np.full(n, np.nan)
    for i in range(n):
        lo_i = max(0, i - W + 1)
        r = ret[lo_i:i+1]; w = v[lo_i:i+1]
        if len(r) >= 6:
            vol[i] = float(np.std(r))
        if len(w) >= 6 and np.std(w) > 0:
            z[i] = (v[i] - w.mean()) / np.std(w)
    out["vol"] = vol
    out["z_vs_ma12"] = z
    return out


SEÑALES = ["mom_3","mom_6","mom_12","ma_cross","macd","donchian",
           "tstat_slope","racha","vol","z_vs_ma12"]

filas = []
g = panel.filter(pl.col("product_id").is_in(elegibles)).sort(["product_id","m"])
for pid, x in g.group_by("product_id", maintain_order=True):
    pid = pid[0] if isinstance(pid, tuple) else pid
    v = x["tn_sa"].to_numpy().astype(float)
    if len(v) < MIN_MESES:
        continue
    sig = señales_de_serie(v)
    fut = np.concatenate([v[HORIZONTE:], np.full(HORIZONTE, np.nan)])
    ret_fut = (fut - v) / np.maximum(np.abs(v), 1e-9)      # cambio relativo a t+2
    for i, m in enumerate(x["m"].to_list()):
        filas.append({"product_id": pid, "m": m, "tn": float(x["tn"][i]),
                      "tn_sa": float(v[i]), "ret_fut": float(ret_fut[i]),
                      **{s: float(sig[s][i]) for s in SEÑALES}})

S = pl.DataFrame(filas)

# OJO: en polars, NaN y null son cosas distintas y drop_nulls() NO saca los NaN.
# Las seniales arrancan con NaN (los primeros meses no tienen ventana suficiente) y
# mom_k puede dar inf si el mes de referencia fue 0. Si no se convierten a null,
# se cuelan en los calculos y todo termina en NaN silenciosamente.
S = S.with_columns([
    pl.when(pl.col(c).is_finite()).then(pl.col(c)).otherwise(None).alias(c)
    for c in SEÑALES + ["ret_fut"]
])

print(f"{S.height:,} observaciones producto-mes con senales")
print()
print("no-nulos por senial (los primeros meses no tienen ventana):")
for c in SEÑALES:
    print(f"   {c:14s} {S[c].drop_nulls().len():>6,}")
print(S.select(["m"] + SEÑALES).describe().head(4))

## 4 — Information Coefficient

El IC es la correlación de rango (Spearman) entre la señal en `t` y el cambio
efectivo a `t+2`. Se calcula **mes a mes** y después se promedia — así el resultado
no lo domina un mes atípico, y el desvío entre meses da el error estándar.

Se usa Spearman y no Pearson porque a la señal le importa el **orden** (qué producto
va a crecer más que otro), no la magnitud exacta, y porque es robusta a los outliers
que abundan en estas series.

Referencias de lectura: en factores de equities, un IC medio de 0,03 ya es
explotable; 0,05 es bueno. Lo que decide no es sólo el nivel sino el **t-stat**:
`IC medio / (desvío del IC / √meses)`. Arriba de 2 se considera señal real.

La última columna es la que importa para el pipe: el **IC residualizado contra
`tstat_slope`**, que es el proxy de `B1`. Mide qué agrega cada señal *por encima* de
la tendencia que el modelo ya conoce.

In [ ]:
def ic_por_mes(df, señal, target="ret_fut", control=None):
    ics = []
    for m, g in df.group_by("m"):
        g = g.drop_nulls([señal, target])
        if g.height < 30:
            continue
        x = g[señal].to_numpy(); y = g[target].to_numpy()
        if np.std(x) == 0 or np.std(y) == 0:
            continue
        if control is not None:
            c = g[control].to_numpy()
            if np.std(c) > 0:                       # residualizar la senial contra el control
                b = np.polyfit(c, x, 1)
                x = x - (b[1] + b[0]*c)
                if np.std(x) == 0:
                    continue
        ics.append(float(stats.spearmanr(x, y).statistic))
    if not ics:
        return np.nan, np.nan, 0
    ics = np.array(ics)
    t = ics.mean() / (ics.std(ddof=1)/np.sqrt(len(ics))) if len(ics) > 1 and ics.std(ddof=1) > 0 else np.nan
    return float(ics.mean()), float(t), len(ics)


filas = []
for s in SEÑALES:
    ic, t, n = ic_por_mes(S, s)
    ic_r, t_r, _ = ic_por_mes(S, s, control="tstat_slope") if s != "tstat_slope" else (np.nan, np.nan, 0)
    filas.append({"señal": s, "IC": round(ic,4), "t_stat": round(t,2), "meses": n,
                  "IC_resid_vs_tendencia": None if np.isnan(ic_r) else round(ic_r,4),
                  "t_resid": None if np.isnan(t_r) else round(t_r,2)})

IC = pl.DataFrame(filas).with_columns(pl.col("IC").abs().alias("_abs")).sort("_abs", descending=True).drop("_abs")
print(IC)

In [ ]:
d = IC.drop_nulls("IC").filter(pl.col("IC").is_finite()).sort("IC")
descartadas = IC.height - d.height
if descartadas:
    print(f"{descartadas} senial(es) sin IC calculable (varianza cero o pocos meses), "
          f"se omiten del grafico")
ys = list(range(d.height))
vals = d["IC"].to_numpy()
cols = [POS if v >= 0 else NEG for v in vals]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(ys, vals, color=cols, height=.62)
for i, (v, t) in enumerate(zip(vals, d["t_stat"].to_list())):
    et = f"{v:+.3f}   t={t:.1f}" if t is not None and np.isfinite(t) else f"{v:+.3f}"
    ax.annotate(et, (v, i), xytext=(6 if v >= 0 else -6, 0), textcoords="offset points",
                ha="left" if v >= 0 else "right", va="center", color=TINTA2, fontsize=8)
ax.axvline(0, color=EJE_C, linewidth=1.2)
ax.set_yticks(ys); ax.set_yticklabels(d["señal"].to_list())
lim = float(np.nanmax(np.abs(vals))) * 1.7 if len(vals) else 1.0
if not np.isfinite(lim) or lim <= 0:
    lim = 1.0
ax.set_xlim(-lim, lim)
limpiar(ax, f"Information Coefficient contra el cambio a {HORIZONTE} meses", x="IC (Spearman)")
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
plt.tight_layout(); plt.show()

print("Azul = la senial predice cambios positivos.  Rojo = predice a la baja")
print("(una senial con IC negativo es igual de util: se usa con el signo dado vuelta).")
print("|t| > 2 se considera senial real y no ruido.")

## 5 — ¿Baja el WAPE?

El IC dice si hay orden predictivo. Esto dice si sirve para el problema real.

Se toma el naive de horizonte 2 y se lo **inclina** con la señal:

```
predicción = tn(t) × (1 + k · señal_estandarizada)
```

`k` se ajusta sobre los primeros dos tercios de los meses y se evalúa en el último
tercio. Es una partición temporal, igual que el pipe: sin eso, "ajustar k" y "medir"
sobre los mismos datos daría una mejora garantizada y falsa.

In [ ]:
meses_ord = sorted(S["m"].unique().to_list())
corte = meses_ord[int(len(meses_ord)*2/3)]
print(f"ajuste de k: meses <= {m_a_periodo(corte)}   evaluacion: meses > {m_a_periodo(corte)}")

def wape(real, pred):
    real = np.asarray(real,float); pred = np.maximum(np.asarray(pred,float), 0)
    den = np.abs(real).sum()
    return float(np.abs(real-pred).sum()/den) if den else np.nan

D = S.drop_nulls(["ret_fut"]).with_columns(
        (pl.col("tn_sa")*(1+pl.col("ret_fut"))).alias("real_fut_sa"))
tr = D.filter(pl.col("m") <= corte); te = D.filter(pl.col("m") > corte)

base_tr = wape(tr["real_fut_sa"], tr["tn_sa"])
base_te = wape(te["real_fut_sa"], te["tn_sa"])
print(f"\nnaive               train {base_tr:.4f}   test {base_te:.4f}")

res = []
for s in SEÑALES:
    a = tr.drop_nulls(s); b = te.drop_nulls(s)
    if a.height < 500 or b.height < 200:
        continue
    mu, sd = a[s].mean(), a[s].std()
    if not sd or sd == 0:
        continue
    zt = ((a[s]-mu)/sd).to_numpy().clip(-3,3)
    ze = ((b[s]-mu)/sd).to_numpy().clip(-3,3)
    mejor_k, mejor_w = 0.0, wape(a["real_fut_sa"], a["tn_sa"])
    for k in np.arange(-.30, .31, .01):
        w = wape(a["real_fut_sa"], a["tn_sa"].to_numpy()*(1+k*zt))
        if w < mejor_w:
            mejor_k, mejor_w = float(k), w
    w_te = wape(b["real_fut_sa"], b["tn_sa"].to_numpy()*(1+mejor_k*ze))
    res.append({"señal": s, "k_ajustado": round(mejor_k,3),
                "wape_train": round(mejor_w,4), "wape_test": round(w_te,4),
                "mejora_test_%": round(100*(base_te-w_te)/base_te, 2)})

R = pl.DataFrame(res).sort("mejora_test_%", descending=True)
print(R)

if R.height:
    mejor = R.row(0, named=True)
    print(f"\nVEREDICTO")
    if mejor["mejora_test_%"] > 1:
        print(f"  '{mejor['señal']}' mejora el naive {mejor['mejora_test_%']:.1f}% en test.")
        print("  Vale la pena pasarla al pipe como feature.")
    else:
        print(f"  Ninguna senial mejora el naive mas de 1% en test")
        print(f"  (la mejor, '{mejor['señal']}', da {mejor['mejora_test_%']:+.1f}%).")
        print("  Ojo: esto NO las descarta como features. Acá se las usa de una forma")
        print("  muy rigida -- un unico k lineal para todos los productos. Un GBM puede")
        print("  usarlas condicionadas al resto del contexto, que es bastante mas potente.")
        print("  El IC es mejor guia que este backtest para decidir cuales pasar al pipe.")

## 6 — Exportar las señales

In [ ]:
export = (S.with_columns(pl.col("m").map_elements(m_a_periodo, return_dtype=pl.Int64).alias("periodo"))
           .select(["product_id","periodo"] + SEÑALES))

out = DIR_OUT / "features_trend_following.parquet"
export.write_parquet(out)
print(f"Guardado: {out}")
print(f"{export.height:,} filas x {export.width} columnas")
print(f"\nSeniales: {SEÑALES}")
print("\nTodas son causales: usan solo datos hasta el mes de la fila.")
print("`ret_fut` NO se exporta: es el target del analisis, seria leakage directo.")

### Cómo llevarlo al pipe

En `02_FE`, después de armar `df_norm`:

```python
trend = pl.read_parquet(RUTA_FE / "features_trend_following.parquet")
df_norm = df_norm.join(trend, on=["product_id", "periodo"], how="left")
```

Y en `03_Optuna` corré con `'sufijo': 'conTrend'` para que no se pise con el
experimento base en el leaderboard.

**Cuáles pasar**: guiate por el `IC_resid_vs_tendencia` de la sección 4, no por el
IC crudo. Una señal con IC alto pero residual cerca de cero está diciendo lo mismo
que `B1`, que el modelo ya tiene — sumarla agrega colinealidad sin información.

Un detalle sobre la desestacionalización: las señales salen de `tn_sa`, calculada con
un índice estacional estimado sobre **todo** el período. Es una forma leve de
leakage — el índice de enero usa datos de todos los eneros, incluido el futuro. Con
un índice de mercado agregado y estable el efecto es chico, pero si querés ser
estricto, recalculá `IDX` usando sólo meses anteriores al corte de train, igual que
en los clusters de DTW.